# 04 — Module D: Loss Amount Regression (Outlier-Heavy)
Predict how much money is lost on defaulted loans.

In [ ]:
import sys, os

# Add project root to path
PROJECT_ROOT = "/Users/nando/PycharmProjects/PythonProject/SmartLender"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from xgboost import XGBRegressor

from src.data.loader import load_raw_data, split_data
from src.data.preprocessor import build_preprocessor, get_catboost_cat_indices
from src.models.registry import get_regressors
from src.models.trainer import run_arena, train_and_evaluate
from src.evaluation.metrics import regression_metrics
from src.evaluation.comparison import save_comparison
from src.config import *

%matplotlib inline
mlflow.set_experiment('SmartLend')

## Load Data — Defaults Only

In [ ]:
df = load_raw_data(sample_size=200000)

# Filter to only defaulted loans
df_defaults = df[df['loan_status_binary'] == 1].copy()
df_defaults = df_defaults.dropna(subset=['charged_off_amount'])
print(f"Defaulted loans: {len(df_defaults)}")
print(f"Loss amount stats:\n{df_defaults['charged_off_amount'].describe()}")

## Loss Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(df_defaults['charged_off_amount'], bins=100, color='#e74c3c', alpha=0.7)
ax1.set_title('Distribution of Charged-Off Amount')
ax1.set_xlabel('Loss Amount ($)')
ax1.set_ylabel('Count')

# Log-scale
ax2.hist(np.log1p(df_defaults['charged_off_amount']), bins=100, color='#3498db', alpha=0.7)
ax2.set_title('Log Distribution of Charged-Off Amount')
ax2.set_xlabel('log(1 + Loss Amount)')
ax2.set_ylabel('Count')

plt.tight_layout()
plt.savefig('results/figures/module_d_loss_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n95th percentile: ${df_defaults['charged_off_amount'].quantile(0.95):.0f}")
print(f"99th percentile: ${df_defaults['charged_off_amount'].quantile(0.99):.0f}")

## Split and Preprocess

In [ ]:
X_train, X_test, y_train, y_test = split_data(df_defaults, TARGET_LOSS_AMOUNT)
preprocessor = build_preprocessor(X_train)
cat_indices = get_catboost_cat_indices(X_train)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Run the Arena

In [ ]:
regressors = get_regressors()
comparison_df, results = run_arena(
    models=regressors,
    preprocessor=preprocessor,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    metrics_fn=regression_metrics,
    cat_feature_indices=cat_indices,
    module_name='module_d',
)

comparison_df = comparison_df.sort_values('rmse', ascending=True)
print(comparison_df.to_string(index=False))

## Additional Outlier Metrics

In [ ]:
from sklearn.metrics import median_absolute_error

for result in results:
    model = result['model']
    name = result['algorithm']
    if 'CatBoost' in name:
        X_test_pred = X_test.copy()
        for col in X_test_pred.select_dtypes(include=['object', 'category']).columns:
            X_test_pred[col] = X_test_pred[col].fillna('Missing').astype(str)
        y_pred = model.predict(X_test_pred)
    else:
        y_pred = model.predict(X_test)

    med_ae = median_absolute_error(y_test, y_pred)
    errors = np.abs(y_test - y_pred)
    p95_error = np.percentile(errors, 95)
    print(f"{name:25s} | Median AE: ${med_ae:,.0f} | 95th pctile error: ${p95_error:,.0f}")

## XGBoost with Huber Loss

In [ ]:
print("Training XGBoost with Huber loss (robust to outliers)...")
xgb_huber = XGBRegressor(
    n_estimators=200, random_state=RANDOM_STATE,
    objective='reg:pseudohubererror', verbosity=0
)
huber_result = train_and_evaluate(
    name='XGBoost (Huber)',
    model=xgb_huber,
    preprocessor=preprocessor,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    metrics_fn=regression_metrics,
    module_name='module_d',
    save=False,
)
print(f"XGBoost (Huber) RMSE: {huber_result['rmse']:.2f}, MAE: {huber_result['mae']:.2f}")

# Compare with standard XGBoost
xgb_standard = [r for r in results if r['algorithm'] == 'XGBoost'][0]
print(f"XGBoost (MSE)   RMSE: {xgb_standard['rmse']:.2f}, MAE: {xgb_standard['mae']:.2f}")

## Save Results

In [ ]:
save_comparison(comparison_df, 'module_d')